# Credit Card Fraud Detection: Cleaning, EDA, Isolation Forest vs Random Forest

**Dataset:** `creditcard.csv` (284,807 transactions, 31 columns; `V1`-`V28` are PCA components, plus `Time`, `Amount`, and the label `Class` where 1 = fraud).

**Pipeline**
1. Load and clean the data
2. Exploratory data analysis (EDA)
3. Stratified train / validation / test split
4. Models: Isolation Forest (unsupervised anomaly detection) and Random Forest (supervised)
5. Compare on precision, recall, F1, ROC-AUC, PR-AUC, MCC and confusion matrices
6. Select the best model

**Why not accuracy?** Fraud is ~0.17% of transactions, so a model that predicts "legit" for everything scores ~99.8% accuracy and catches zero fraud. We focus on **PR-AUC, recall, precision and F1**.

In [ ]:
import os, sys, glob, warnings, time
import numpy as np
import pandas as pd
import matplotlib
IN_NOTEBOOK = "ipykernel" in sys.modules
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, matthews_corrcoef, confusion_matrix,
                             precision_recall_curve, roc_curve, classification_report)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
OUT_DIR = "outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def save_show(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, name), dpi=130, bbox_inches="tight")
    if IN_NOTEBOOK:
        plt.show()
    plt.close(fig)

## 1. Load the data
Set `DATA_PATH` if your file lives somewhere else. A `.zip` containing the CSV also works.

In [ ]:
DATA_PATH = next((p for p in ["data/creditcard.csv", "../data/creditcard.csv", "creditcard.csv"]
                  if os.path.exists(p)), "data/creditcard.csv")
if not os.path.exists(DATA_PATH):
    hits = glob.glob("**/creditcard*.csv", recursive=True)
    if hits:
        DATA_PATH = hits[0]
print("Loading:", DATA_PATH)
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

## 2. Data cleaning

In [ ]:
print("=== Data types ===")
print(df.dtypes.value_counts(), "\n")
print("=== Missing values ===")
print("Total missing cells:", int(df.isna().sum().sum()))
print("\n=== Label values ===")
print(df["Class"].value_counts())
assert set(df["Class"].unique()) <= {0, 1}, "Class column has unexpected values"

In [ ]:
# Duplicates: identical rows would end up in both train and test and inflate scores (data leakage).
dups = df.duplicated()
print(f"Duplicate rows: {dups.sum()}  (of which fraud: {df.loc[dups, 'Class'].sum()})")
n_before = len(df)
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f"Rows: {n_before:,} -> {len(df_clean):,}")
print(f"Fraud rate after de-duplication: {df_clean['Class'].mean()*100:.4f}%  "
      f"({df_clean['Class'].sum()} fraud cases)")

In [ ]:
# Sanity checks and outlier profile.
print("Negative amounts:", int((df_clean["Amount"] < 0).sum()))
print("Zero-amount transactions:", int((df_clean["Amount"] == 0).sum()))
print("Infinite values:", int(np.isinf(df_clean.select_dtypes("number")).sum().sum()))

# IQR outlier share per column. We do NOT remove outliers: in fraud detection the outliers ARE the signal,
# and tree models are not sensitive to them.
num_cols = [c for c in df_clean.columns if c != "Class"]
q1, q3 = df_clean[num_cols].quantile(0.25), df_clean[num_cols].quantile(0.75)
iqr = q3 - q1
out_share = ((df_clean[num_cols] < q1 - 1.5 * iqr) | (df_clean[num_cols] > q3 + 1.5 * iqr)).mean() * 100
print("\nColumns with the highest share of IQR outliers (%):")
print(out_share.sort_values(ascending=False).head(8).round(2))

In [ ]:
# Feature engineering
# - Time is "seconds since the first transaction" (two days of data). The raw value cannot generalise to new data,
#   but the hour of day can carry a pattern, so we derive Hour and drop Time.
# - Scaling is not needed for tree-based models, so Amount is kept as is.
df_clean["Hour"] = ((df_clean["Time"] // 3600) % 24).astype(int)
df_model = df_clean.drop(columns=["Time"])
FEATURES = [c for c in df_model.columns if c != "Class"]
print(len(FEATURES), "features:", FEATURES)

## 3. Exploratory data analysis

In [ ]:
# 3.1 Class imbalance
counts = df_model["Class"].value_counts().sort_index()
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=["Legit (0)", "Fraud (1)"], y=counts.values, ax=ax[0], palette=["#4C72B0", "#C44E52"])
ax[0].set_title("Class counts"); ax[0].set_ylabel("Transactions")
for i, v in enumerate(counts.values):
    ax[0].text(i, v, f"{v:,}", ha="center", va="bottom")
ax[1].pie(counts.values, labels=["Legit", "Fraud"], colors=["#4C72B0", "#C44E52"],
          autopct="%1.3f%%", startangle=90, explode=(0, 0.15))
ax[1].set_title("Class share")
save_show(fig, "01_class_balance.png")
print(f"Imbalance ratio: 1 fraud per {counts[0]/counts[1]:.0f} legitimate transactions")

In [ ]:
# 3.2 Transaction amount
print(df_model.groupby("Class")["Amount"].describe().round(2))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for cls, color, label in [(0, "#4C72B0", "Legit"), (1, "#C44E52", "Fraud")]:
    ax[0].hist(np.log1p(df_model.loc[df_model.Class == cls, "Amount"]), bins=50, alpha=0.6,
               color=color, label=label, density=True)
ax[0].set_title("log(1 + Amount) by class (normalised)"); ax[0].legend()
sns.boxplot(data=df_model, x="Class", y="Amount", ax=ax[1], showfliers=False, palette=["#4C72B0", "#C44E52"])
ax[1].set_title("Amount by class (outliers hidden)")
sns.boxplot(data=df_model, x="Class", y=np.log1p(df_model["Amount"]), ax=ax[2], palette=["#4C72B0", "#C44E52"])
ax[2].set_title("log(1 + Amount) by class")
save_show(fig, "02_amount.png")

In [ ]:
# 3.3 Time of day
hour_stats = df_model.groupby("Hour")["Class"].agg(["count", "sum", "mean"])
hour_stats["fraud_rate_%"] = hour_stats["mean"] * 100
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].bar(hour_stats.index, hour_stats["count"], color="#4C72B0")
ax[0].set_title("Transactions per hour of day"); ax[0].set_xlabel("Hour")
ax[1].bar(hour_stats.index, hour_stats["fraud_rate_%"], color="#C44E52")
ax[1].set_title("Fraud rate (%) per hour of day"); ax[1].set_xlabel("Hour")
save_show(fig, "03_hour.png")
print("Hours with the highest fraud rate:")
print(hour_stats.sort_values("fraud_rate_%", ascending=False).head(5)[["count", "sum", "fraud_rate_%"]].round(3))

In [ ]:
# 3.4 Which features separate fraud from legit?
corr_with_class = df_model[FEATURES + ["Class"]].corr()["Class"].drop("Class").sort_values()
fig, ax = plt.subplots(figsize=(8, 8))
colors = ["#C44E52" if v < 0 else "#4C72B0" for v in corr_with_class.values]
ax.barh(corr_with_class.index, corr_with_class.values, color=colors)
ax.set_title("Correlation of each feature with Class (fraud)")
save_show(fig, "04_corr_with_class.png")
print("Strongest negative:", corr_with_class.head(5).round(3).to_dict())
print("Strongest positive:", corr_with_class.tail(5).round(3).to_dict())

In [ ]:
# 3.5 Distributions of the most informative features
top_feats = corr_with_class.abs().sort_values(ascending=False).head(8).index.tolist()
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, f in zip(axes.ravel(), top_feats):
    sns.kdeplot(df_model.loc[df_model.Class == 0, f], ax=ax, color="#4C72B0", label="Legit", fill=True, alpha=0.3)
    sns.kdeplot(df_model.loc[df_model.Class == 1, f], ax=ax, color="#C44E52", label="Fraud", fill=True, alpha=0.3)
    ax.set_title(f)
axes[0, 0].legend()
save_show(fig, "05_top_feature_distributions.png")

In [ ]:
# 3.6 Correlation heatmap. PCA components are uncorrelated with each other by construction, so the
# interesting part is the relation to Amount, Hour and Class.
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(df_model.corr(), cmap="coolwarm", center=0, ax=ax, cbar_kws={"shrink": 0.7})
ax.set_title("Correlation heatmap")
save_show(fig, "06_correlation_heatmap.png")

In [ ]:
# 3.7 2-D view of the data (PCA is already applied; we plot the two strongest features)
sample = pd.concat([df_model[df_model.Class == 1],
                    df_model[df_model.Class == 0].sample(5000, random_state=RANDOM_STATE)])
f1_, f2_ = top_feats[0], top_feats[1]
fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(data=sample, x=f1_, y=f2_, hue="Class", palette=["#4C72B0", "#C44E52"],
                alpha=0.6, s=18, ax=ax)
ax.set_title(f"{f1_} vs {f2_} (all fraud + 5,000 legit)")
save_show(fig, "07_scatter_top2.png")

## 4. Train / validation / test split
A **stratified** 60/20/20 split keeps the fraud rate identical in every part.
- **Train**: fit the models
- **Validation**: choose the decision threshold
- **Test**: touched once, for the final comparison

We handle imbalance with **class weights** (`class_weight` for Random Forest) instead of resampling. This has no leakage risk and keeps the test data realistic.

In [ ]:
X = df_model[FEATURES]
y = df_model["Class"]
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=RANDOM_STATE)
for name, yy in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name:11s} rows={len(yy):>7,}  fraud={int(yy.sum()):>4}  rate={yy.mean()*100:.4f}%")

## 5. Models and evaluation helpers

In [ ]:
def best_f1_threshold(y_true, scores):
    """Threshold that maximises F1 on the validation set."""
    p, r, t = precision_recall_curve(y_true, scores)
    f1 = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
    return t[np.argmax(f1)]

def evaluate(name, y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "Model": name,
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "ROC-AUC": roc_auc_score(y_true, scores),
        "PR-AUC": average_precision_score(y_true, scores),
        "MCC": matthews_corrcoef(y_true, pred),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Threshold": float(threshold),
    }

results, test_scores, fitted = [], {}, {}

### 5.1 Isolation Forest (unsupervised anomaly-detection baseline)
It never sees the labels during training. It isolates points that are easy to separate from the rest, which is the textbook anomaly-detection approach and gives us a baseline to show how much the supervised models add.

In [ ]:
t0 = time.time()
iso = IsolationForest(n_estimators=200, contamination=float(y_train.mean()),
                      random_state=RANDOM_STATE, n_jobs=-1)
iso.fit(X_train)                      # labels are NOT used
iso_val = -iso.score_samples(X_val)   # higher = more anomalous
iso_test = -iso.score_samples(X_test)
thr = best_f1_threshold(y_val, iso_val)
results.append(evaluate("Isolation Forest (unsupervised)", y_test, iso_test, thr))
test_scores["Isolation Forest (unsupervised)"] = iso_test
fitted["Isolation Forest (unsupervised)"] = iso
print(f"Isolation Forest done in {time.time()-t0:.0f}s")

### 5.2 Random Forest

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=150, max_depth=12, min_samples_leaf=2, max_features="sqrt",
    max_samples=0.5, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
rf_val = rf.predict_proba(X_val)[:, 1]
rf_test = rf.predict_proba(X_test)[:, 1]
thr = best_f1_threshold(y_val, rf_val)
results.append(evaluate("Random Forest", y_test, rf_test, thr))
test_scores["Random Forest"] = rf_test
fitted["Random Forest"] = rf
print(f"Random Forest done in {time.time()-t0:.0f}s, threshold={thr:.3f}")

## 6. Model comparison (on the held-out test set)

In [ ]:
res = pd.DataFrame(results).set_index("Model")
metric_cols = ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC", "MCC"]
display_df = res[metric_cols + ["TP", "FP", "FN", "TN", "Threshold"]].round(4)
res.to_csv(os.path.join(OUT_DIR, "model_comparison.csv"))
display_df

In [ ]:
# Bar chart of all metrics
fig, ax = plt.subplots(figsize=(11, 5))
res[metric_cols].plot(kind="bar", ax=ax, colormap="viridis", edgecolor="black", width=0.8)
ax.set_ylim(0, 1.05); ax.set_ylabel("Score"); ax.set_title("Model comparison on the test set")
plt.xticks(rotation=15, ha="right"); ax.legend(loc="lower right", ncol=3)
save_show(fig, "08_metric_comparison.png")

In [ ]:
# PR and ROC curves
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for name, s in test_scores.items():
    p, r, _ = precision_recall_curve(y_test, s)
    ax[0].plot(r, p, label=f"{name} (AP={average_precision_score(y_test, s):.3f})")
    fpr, tpr, _ = roc_curve(y_test, s)
    ax[1].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, s):.3f})")
ax[0].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall curve"); ax[0].legend()
ax[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
ax[1].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curve"); ax[1].legend()
save_show(fig, "09_pr_roc_curves.png")

In [ ]:
# Confusion matrices at each model's validation-tuned threshold
fig, axes = plt.subplots(1, len(res), figsize=(5.2 * len(res), 4.4))
axes = np.atleast_1d(axes)
for ax, (name, row) in zip(axes, res.iterrows()):
    cm = np.array([[row.TN, row.FP], [row.FN, row.TP]]).astype(int)
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Pred legit", "Pred fraud"], yticklabels=["Actual legit", "Actual fraud"])
    ax.set_title(name, fontsize=11)
save_show(fig, "10_confusion_matrices.png")

In [ ]:
# Feature importance
imp_models = ["Random Forest"]
fig, axes = plt.subplots(1, len(imp_models), figsize=(7 * len(imp_models), 6))
axes = np.atleast_1d(axes)
for ax, m in zip(axes, imp_models):
    imp = pd.Series(fitted[m].feature_importances_, index=FEATURES).sort_values().tail(15)
    ax.barh(imp.index, imp.values, color="#55A868")
    ax.set_title(f"{m}: top 15 features")
save_show(fig, "11_feature_importance.png")

## 7. Choosing the best model
**Selection rule:** the primary criterion is **PR-AUC** (it is threshold-independent and honest under heavy imbalance); ties are broken by **F1**. We also report recall and precision, because in practice you trade missed fraud (false negatives) against false alarms (false positives).

In [ ]:
ranked = res.sort_values(["PR-AUC", "F1"], ascending=False)
best_name = ranked.index[0]
b = ranked.loc[best_name]
print("Ranking (all models):")
print(ranked[["PR-AUC", "F1", "Recall", "Precision", "ROC-AUC", "MCC"]].round(4), "\n")
print(f"BEST MODEL: {best_name}")
print(f"  It catches {int(b.TP)} of {int(b.TP + b.FN)} fraud cases in the test set (recall {b.Recall:.1%}), "
      f"with {int(b.FP)} false alarms among {int(b.FP + b.TN):,} legitimate transactions (precision {b.Precision:.1%}).")
if len(ranked) > 1:
    o = ranked.iloc[1]
    print(f"  Runner-up {ranked.index[1]}: PR-AUC {o['PR-AUC']:.4f} vs {b['PR-AUC']:.4f}, F1 {o['F1']:.4f} vs {b['F1']:.4f}.")
print(f"\nClassification report for {best_name} (test set):")
best_pred = (test_scores[best_name] >= b.Threshold).astype(int)
print(classification_report(y_test, best_pred, target_names=["Legit", "Fraud"], digits=4))

joblib.dump({"model": fitted[best_name], "features": FEATURES, "threshold": float(b.Threshold)},
            os.path.join(OUT_DIR, "best_model.joblib"))
print("Saved:", os.path.join(OUT_DIR, "best_model.joblib"))

## 8. Caveats
- The test set contains only ~**95 fraud cases**, so metrics move noticeably with a different random seed. For a more robust estimate, repeat with several seeds or use stratified k-fold CV.
- The `V1`-`V28` features are anonymised PCA outputs, so feature importance cannot be interpreted in business terms.
- The data covers two days of transactions from European cardholders (2013). Real deployments need monitoring for drift and periodic retraining.
- To squeeze out more performance, tune hyperparameters with `RandomizedSearchCV(scoring="average_precision")` and consider SMOTE inside a cross-validation pipeline.